# Experimento 3 - Sensibilidade ao limiar minimo

Objetivo: avaliar o impacto de `qkd_min_bits_threshold`, parametro explicito do enlace usado pelas politicas `threshold` e `hybrid`.

Contexto: o enlace inicia com `qkd_min_bits_threshold = 128`, e esse valor aparece no estado monitorado do link.

Este experimento varre os limiares:
- 32
- 64
- 128
- 256

e executa para as politicas:
- threshold
- hybrid

In [8]:
import sys
import os
import random
import importlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '/home/esdras/Wqnets/QuantumNet')

# Força reload do módulo (útil quando você editou o código e o kernel já estava rodando)
import quantumnet.experiments.qkd_policy_experiments as _qkdexp
import quantumnet.experiments as _qexp
importlib.reload(_qkdexp)
importlib.reload(_qexp)

from quantumnet.experiments import (
    stable_seed,
    run_trials,
    aggregate_trials,
    write_csv,
 )

print('Imports carregados com sucesso')

Imports carregados com sucesso


## Configuracao

Carga fixa (intermediaria), para isolar o efeito do limiar:
- topologia: Linha(4)
- enlaces avaliados: (0,1), (1,2), (2,3)
- requisicoes por enlace: 8
- tamanhos possiveis por requisicao (bits): 24, 32, 48, 64

In [ ]:
policies = ['threshold', 'hybrid']
threshold_values = [32, 64, 96, 128, 192, 256]
link_pairs = [(0, 1), (1, 2), (2, 3)]

# Cenário de carga fixo (representativo) para isolar o efeito do limiar
fixed_load_profile = {
    'requests_per_link': 30,
    'bit_options': [24, 32, 48, 64],
}

buffer_capacity_bits = 512

# Modo validação (rápido)
trials_per_setting = 5
base_seed = 20260326

print('Políticas:', policies)
print('Limiares:', threshold_values)
print('Repetições por configuração:', trials_per_setting)
print('Buffer (capacidade por enlace):', buffer_capacity_bits)

Políticas: ['threshold', 'hybrid']
Limiares: [32, 64, 96, 128, 192, 256]
Repetições por configuração: 5
Buffer (capacidade por enlace): 512


## Execucao do experimento

Metricas observadas por repeticao:
- taxa de atendimento (`service_rate`) e taxa de falha/negação (`denial_rate`)
- eficiencia (`efficiency`)
- eventos de reposicao (`replenishment_events`)
- utilizacao media do buffer (`buffer_util_mean`)
- `bits_available` final (soma nos enlaces)

In [ ]:
def build_fixed_schedule(trial_id: int) -> tuple[int, list[tuple[int, int, int]]]:
    schedule_seed = stable_seed(base_seed, 'exp3_schedule', trial_id)
    rng = random.Random(schedule_seed)
    schedule: list[tuple[int, int, int]] = []
    for alice_id, bob_id in link_pairs:
        for _ in range(int(fixed_load_profile['requests_per_link'])):
            schedule.append((alice_id, bob_id, int(rng.choice(fixed_load_profile['bit_options']))))
    return int(schedule_seed), schedule


mp_start_method = 'spawn'  # mais estável em notebooks
n_jobs = max(1, min(8, (os.cpu_count() or 2) - 1))
print(f"Multiprocessing: n_jobs={n_jobs} start_method={mp_start_method}")

tasks = []
for trial in range(1, trials_per_setting + 1):
    schedule_seed, request_schedule = build_fixed_schedule(trial)
    for policy in policies:
        for min_threshold in threshold_values:
            tasks.append({
                'policy': policy,
                'trial_id': trial,
                'base_seed': base_seed,
                'topology_name': 'Linha',
                'topology_nodes': 4,
                'link_pairs': link_pairs,
                'request_schedule': request_schedule,
                'min_threshold_bits': int(min_threshold),
                'buffer_capacity_bits': buffer_capacity_bits,
                'extra': {
                    'experiment': 'exp3',
                    'min_threshold': int(min_threshold),
                    'buffer_capacity_bits': buffer_capacity_bits,
                    'schedule_seed': int(schedule_seed),
                    'schedule_len': len(request_schedule),
                },
            })

rows = run_trials(tasks, n_jobs=n_jobs, mp_start_method=mp_start_method)

results_df = pd.DataFrame(rows)
display(results_df.sort_values(['policy', 'min_threshold', 'trial']).reset_index(drop=True))

summary_rows = aggregate_trials(
    rows,
    group_keys=['policy', 'min_threshold'],
    metric_keys=['service_rate', 'denial_rate', 'buffer_util_mean', 'replenishment_events', 'efficiency'],
 )
summary_df = pd.DataFrame(summary_rows).sort_values(['policy', 'min_threshold']).reset_index(drop=True)

print('\nResumo (média ± desvio padrão)')
display(summary_df)

# Exporta CSVs
out_dir = Path('results') / f'validate_{trials_per_setting}'
out_dir.mkdir(parents=True, exist_ok=True)
write_csv(str(out_dir / 'exp3_trials.csv'), rows)
write_csv(str(out_dir / 'exp3_summary.csv'), summary_rows)
print(f"CSVs salvos em: {out_dir.resolve()}")

# Gráficos de sensibilidade (média ± std)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

plots = [
    ('service_rate', 'Taxa de atendimento'),
    ('denial_rate', 'Taxa de falha/negação'),
    ('buffer_util_mean', 'Utilização média do buffer'),
    ('replenishment_events', 'Eventos de reposição'),
 ]

for ax, (metric, title) in zip(axes, plots):
    for policy in policies:
        subset = summary_df[summary_df['policy'] == policy]
        x = subset['min_threshold']
        y = subset[f'{metric}_mean']
        yerr = subset[f'{metric}_std']
        ax.errorbar(x, y, yerr=yerr, marker='o', capsize=4, label=policy)
    ax.set_title(title)
    ax.set_xlabel('Limiar mínimo (bits)')
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.legend(title='Política')

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

## Resultado detalhado por repeticao

In [ ]:
display(results_df.sort_values(['policy', 'min_threshold', 'trial']).reset_index(drop=True))

,policy,min_threshold,trial,total_generated_bits,total_consumed_bits,served_requests,denied_requests,replenishment_events,bits_available,service_rate,efficiency,instant_shortage_events
0,hybrid,32,1,920,912,22,2,30,8,0.916667,0.991304,24
1,hybrid,32,2,776,768,20,4,28,8,0.833333,0.989691,24
2,hybrid,32,3,936,928,22,2,34,8,0.916667,0.991453,24
3,hybrid,32,4,864,864,21,3,32,0,0.875000,1.000000,23
4,hybrid,32,5,696,664,20,4,23,32,0.833333,0.954023,24
...,...,...,...,...,...,...,...,...,...,...,...,...
75,threshold,256,6,0,0,0,24,0,0,0.000000,0.000000,24
76,threshold,256,7,0,0,0,24,0,0,0.000000,0.000000,24
77,threshold,256,8,256,256,6,18,1,0,0.250000,1.000000,19
78,threshold,256,9,0,0,0,24,0,0,0.000000,0.000000,24


## Resumo agregado

Leitura principal (média ± desvio):
- taxa de atendimento (`service_rate`)
- taxa de falha/negação (`denial_rate`)
- utilização média do buffer (`buffer_util_mean`)
- número de eventos de reposição (`replenishment_events`)
- eficiência (`efficiency`)

In [ ]:
# (Opcional) Reexibir o mesmo resumo calculado acima
display(summary_df)

,policy,min_threshold,service_rate_mean,service_rate_std,efficiency_mean,replenishment_events_mean,bits_available_mean,instant_shortage_events_mean,denied_requests_mean,served_requests_mean
0,hybrid,32,0.845833,0.076199,0.988807,29.0,8.8,23.7,3.7,20.3
1,hybrid,64,0.820833,0.044140,0.964399,17.6,31.2,21.5,4.3,19.7
2,hybrid,128,0.666667,0.070820,1.000000,14.5,0.0,22.4,8.0,16.0
3,hybrid,256,0.654167,0.122742,1.000000,14.9,0.0,23.2,8.3,15.7
4,threshold,32,0.550000,0.078075,0.932455,13.2,34.4,24.0,10.8,13.2
5,threshold,64,0.270833,0.111544,0.856667,4.4,40.8,21.9,17.5,6.5
6,threshold,128,0.087500,0.074665,0.571875,0.8,16.8,22.7,21.9,2.1
7,threshold,256,0.041667,0.090010,0.178125,0.2,5.6,23.2,23.0,1.0


## Leitura interpretativa automatica

Padroes esperados:
- limiar maior tende a aumentar `replenishment_events`
- limiar menor tende a aumentar risco de falta instantanea
- `hybrid` tende a ficar mais equilibrado em carga intermediaria por combinar manutencao de estoque e reposicao por necessidade

In [ ]:
print('Resumo por politica:')
for policy in policies:
    subset = summary_df[summary_df['policy'] == policy].sort_values('min_threshold')
    print(f'\nPolitica: {policy}')
    for _, row in subset.iterrows():
        util = row.get('buffer_util_mean_mean', None)
        util_str = 'n/a' if util is None or pd.isna(util) else f"{float(util):.3f}"
        print(
            f"  limiar={int(row['min_threshold']):3d} | "
            f"atendimento={row['service_rate_mean']:.3f}±{row['service_rate_std']:.3f} | "
            f"falha={row['denial_rate_mean']:.3f}±{row['denial_rate_std']:.3f} | "
            f"reposicoes={row['replenishment_events_mean']:.1f}±{row['replenishment_events_std']:.1f} | "
            f"util_buffer={util_str}"
        )

print('\nPivot (service_rate_mean):')
pivot_compare = summary_df.pivot(index='min_threshold', columns='policy', values='service_rate_mean')
display(pivot_compare)

Resumo por politica:

Politica: threshold
  limiar= 32 | atendimento=0.550 | eficiencia=0.932 | reposicoes=13.2 | falta_instantanea=24.0 | buffer_final=34.4
  limiar= 64 | atendimento=0.271 | eficiencia=0.857 | reposicoes=4.4 | falta_instantanea=21.9 | buffer_final=40.8
  limiar=128 | atendimento=0.087 | eficiencia=0.572 | reposicoes=0.8 | falta_instantanea=22.7 | buffer_final=16.8
  limiar=256 | atendimento=0.042 | eficiencia=0.178 | reposicoes=0.2 | falta_instantanea=23.2 | buffer_final=5.6

Politica: hybrid
  limiar= 32 | atendimento=0.846 | eficiencia=0.989 | reposicoes=29.0 | falta_instantanea=23.7 | buffer_final=8.8
  limiar= 64 | atendimento=0.821 | eficiencia=0.964 | reposicoes=17.6 | falta_instantanea=21.5 | buffer_final=31.2
  limiar=128 | atendimento=0.667 | eficiencia=1.000 | reposicoes=14.5 | falta_instantanea=22.4 | buffer_final=0.0
  limiar=256 | atendimento=0.654 | eficiencia=1.000 | reposicoes=14.9 | falta_instantanea=23.2 | buffer_final=0.0

Comparacao direta threshol

policy,hybrid,threshold
min_threshold,,
32,0.845833,0.550000
64,0.820833,0.270833
128,0.666667,0.087500
256,0.654167,0.041667


## O que este experimento responde

Mostra a sensibilidade das politicas `threshold` e `hybrid` ao parametro `qkd_min_bits_threshold` e em que regime o ajuste de limiar melhora estabilidade (atendimento) ao custo de maior reposicao.

Tambem evidencia o trade-off entre manter estoque (menos falta instantanea) e aumentar pressao de reposicao no sistema.